# 03 · News + Llama fusion (Colab T4)

Runtime → Change runtime type → **T4 GPU**. Colab Secrets needed: `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, `HF_TOKEN`.
After any disconnect: rerun **Setup**, then the step you were on (all steps resume).

## Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'
KRONOS = '/content/Kronos'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/kronos_agent/phase3.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

!test -d $KRONOS || git clone -q https://github.com/shiyu-coder/Kronos $KRONOS
%pip install -q einops safetensors ccxt yfinance bitsandbytes accelerate
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)

# Secrets -> environment (inherited by the scripts below). Never print them.
for k in ('ALPACA_API_KEY', 'ALPACA_SECRET_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)

# BTC forecasts that exist: lookback 64, 2025 H1
BTC = "--set lookback=64 --set windows.pre_cutoff=null --set 'windows.post_cutoff=[2025-01-01,2025-07-01]'"
!nvidia-smi -L
print(ROOT, CODE, KRONOS)

## Step 1 · Fetch news + show one prompt (CPU, ~2–5 min first time; no LLM)

In [ ]:
!cd $CODE && python scripts/run_llm.py --config configs/spy_1d.yaml --root $ROOT --dry-run
!cd $CODE && python scripts/run_llm.py --config configs/btc_1h.yaml --root $ROOT --dry-run $BTC

## Step 2 · Llama smoke test: 8 SPY origins (first run downloads ~16 GB, ~5–10 min)

In [ ]:
!cd $CODE && python scripts/run_llm.py --config configs/spy_1d.yaml --root $ROOT --variant real --limit 8
import pandas as pd
from kronos_agent.config import load_config, run_tag
from kronos_agent.store import ResultStore
t = run_tag(load_config(f'{CODE}/configs/spy_1d.yaml'))
d = ResultStore(f'{ROOT}/llm/{t}_real').load()
print(d[['origin_ts', 'n_heads', 'letter_mass', 'n_tokens']].to_string(index=False))
print(pd.DataFrame(d['q'].tolist(), columns=list('ABCDEFG')).round(3).to_string(index=False))

## Step 3 · Full scoring: SPY (≈815 origins × 3 variants) and BTC (181 × 3)

In [ ]:
for v in ('real', 'shuffled', 'anon'):
    !cd $CODE && python scripts/run_llm.py --config configs/spy_1d.yaml --root $ROOT --variant $v

In [ ]:
for v in ('real', 'shuffled', 'anon'):
    !cd $CODE && python scripts/run_llm.py --config configs/btc_1h.yaml --root $ROOT --variant $v $BTC

## Step 4 · Fusion report (CPU, minutes)

In [ ]:
!cd $CODE && python scripts/run_phase3.py --config configs/spy_1d.yaml --root $ROOT
!cd $CODE && python scripts/run_phase3.py --config configs/btc_1h.yaml --root $ROOT $BTC